# fourier

Historical experiment source from the internship project. Outputs and personal runtime metadata are cleared. This version is retained for provenance; see the root README for the maintained entry points. Colab Drive paths describe the original environment and require your own matching inputs. Source: `fourier.py`. No fresh GPU execution is claimed.

In [ ]:
# -*- coding: utf-8 -*-
"""Fourier.ipynb

Automatically generated by Colab.

Original file is located at
    [original private Colab link omitted]
"""

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import pandas as pd
import numpy as np

CSV_FOLDER = "/content/drive/MyDrive/Fourier/ExtractedCSV"

csv_files = sorted(

    glob.glob(

        os.path.join(CSV_FOLDER, "*.csv")

    )

)

print("Total CSV :", len(csv_files))

master_objects = set()

for file in csv_files:

    try:
        df = pd.read_csv(file)

        if "Object" not in df.columns:
            continue

        master_objects.update(df["Object"].tolist())

    except pd.errors.EmptyDataError:
        print(f"Skipped empty file: {file}")

master_objects = sorted(master_objects)

print(master_objects)
print("Total Objects :", len(master_objects))

master_df = pd.DataFrame(

    {

        "Object": master_objects

    }

)

master_df.to_csv(

    "/content/drive/MyDrive/Fourier/master_object_list.csv",

    index=False

)

import os
import glob
import numpy as np
import pandas as pd

CSV_FOLDER = "/content/drive/MyDrive/Fourier/ExtractedCSV"

MASTER_FILE = "/content/drive/MyDrive/Fourier/master_object_list.csv"

OUTPUT_FOLDER = "/content/drive/MyDrive/Fourier/SceneVectors"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

master_df = pd.read_csv(MASTER_FILE)

master_objects = master_df["Object"].tolist()

print(master_objects)

print(len(master_objects))

csv_files = sorted(

    glob.glob(

        os.path.join(CSV_FOLDER, "*.csv")

    )

)

print(len(csv_files))

scene_vectors = {}

for csv_file in csv_files:

    room_name = os.path.splitext(

        os.path.basename(csv_file)

    )[0]

    df = pd.read_csv(csv_file)
    if "Object" not in df.columns:

      df = pd.DataFrame(

        columns=["Object","X","Y","Z"]

     )

    room_vector = []

    for obj in master_objects:

        row = df[

            df["Object"] == obj

        ]

        if len(row) == 0:

            room_vector.extend(

                [0.0,0.0,0.0]

            )

        else:

            room_vector.extend(

                [

                    float(row.iloc[0]["X"]),

                    float(row.iloc[0]["Y"]),

                    float(row.iloc[0]["Z"])

                ]

            )

    scene_vectors[room_name] = np.array(

        room_vector,

        dtype=np.float32

    )

print(scene_vectors.keys())

print(scene_vectors["fullyfurnishedroom"].shape)

print(scene_vectors["fullyfurnishedroom"])

for room in scene_vectors:

    print(

        room,

        scene_vectors[room].shape

    )

for room in scene_vectors:

    vector_df = pd.DataFrame(

        {

            "Value": scene_vectors[room]

        }

    )

    vector_df.to_csv(

        os.path.join(

            OUTPUT_FOLDER,

            room + "_vector.csv"

        ),

        index=False

    )

import pickle

with open(

    "/content/drive/MyDrive/Fourier/scene_vectors.pkl",

    "wb"

) as f:

    pickle.dump(scene_vectors,f)

import numpy as np

import pandas as pd

from scipy.spatial.distance import cdist

num_objects = len(master_objects)

print(num_objects)

coordinates = scene_vectors["fullyfurnishedroom"].reshape(

    num_objects,

    3

)

print(coordinates.shape)

distance_matrix = cdist(

    coordinates,

    coordinates,

    metric="euclidean"

)

print(distance_matrix.shape)

print(distance_matrix)

distance_embeddings = {}

for room in scene_vectors:

    coords = scene_vectors[room].reshape(

        num_objects,

        3

    )

    dist = cdist(

        coords,

        coords,

        metric="euclidean"

    )

    distance_embeddings[room] = dist

for room in distance_embeddings:

    print(

        room,

        distance_embeddings[room].shape

    )

distance_vectors = {}

for room in distance_embeddings:

    distance_vectors[room] = (

        distance_embeddings[room]

        .flatten()

        .astype(np.float32)

    )

for room in distance_vectors:

    print(
        room,
        np.isnan(distance_vectors[room]).sum()
    )

print(

    distance_vectors["fullyfurnishedroom"].shape

)

room_names = sorted(distance_vectors.keys())

embedding_matrix = np.stack(
    [distance_vectors[r] for r in room_names]
)

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(

    embedding_matrix

)

similarity_df = pd.DataFrame(

    similarity_matrix,

    index=room_names,

    columns=room_names

)

similarity_df

for room in distance_embeddings:
    print(room, np.isnan(distance_embeddings[room]).any())

for room in distance_vectors:
    print(room, np.linalg.norm(distance_vectors[room]))

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))

sns.heatmap(similarity_df, annot=True, cmap="viridis")

plt.title(

    "Distance Matrix Room Similarity Matrix",

    fontsize=16

)

plt.xlabel("Rooms")

plt.ylabel("Rooms")

plt.tight_layout()
plt.show()

plt.figure(figsize=(12,10))

sns.heatmap(

    similarity_df,

    annot=True,

    cmap="viridis",

    fmt=".3f",

    square=True,

    linewidths=0.5

)

plt.title(

    "Distance Matrix Similarity Matrix"

)

plt.tight_layout()

plt.savefig(

"/content/drive/MyDrive/Fourier/DistanceMatrix_Heatmap.png",

dpi=300

)

plt.close()